# Spam SMS Detection - Google Colab

This notebook trains SMS spam detection models using TF-IDF text features.

## 1. Upload Dataset Zip

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_path = next(iter(uploaded.keys()))
zip_path

## 2. Extract Dataset

In [ ]:
from pathlib import Path
import zipfile

Path('data').mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    for name in z.namelist():
        if name.endswith('spam.csv'):
            z.extract(name, 'data')

for file in Path('data').rglob('spam.csv'):
    target = Path('data') / 'spam.csv'
    if file != target:
        target.write_bytes(file.read_bytes())

list(Path('data').glob('*.csv'))

## 3. Train Models

In [ ]:
import joblib
import pandas as pd
import re
from pathlib import Path
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

def clean_text(message):
    message = str(message).lower()
    message = re.sub(r'http\S+|www\S+', ' url ', message)
    message = re.sub(r'\d+', ' number ', message)
    message = re.sub(r'[^a-z\s]', ' ', message)
    return re.sub(r'\s+', ' ', message).strip()

df = pd.read_csv('data/spam.csv', encoding='latin-1')
df = df[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'message'})
df['spam'] = df['label'].map({'ham': 0, 'spam': 1})
df['clean_message'] = df['message'].map(clean_text)

x_train, x_test, y_train, y_test = train_test_split(
    df['clean_message'], df['spam'], test_size=0.2, random_state=42, stratify=df['spam']
)

def tfidf():
    return TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, max_df=0.95)

models = {
    'naive_bayes': MultinomialNB(alpha=0.2),
    'logistic_regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'linear_svm': CalibratedClassifierCV(estimator=LinearSVC(class_weight='balanced', random_state=42), cv=3),
}

rows = []
trained = {}
for name, clf in models.items():
    pipe = Pipeline([('tfidf', tfidf()), ('model', clf)])
    pipe.fit(x_train, y_train)
    pred = pipe.predict(x_test)
    prob = pipe.predict_proba(x_test)[:, 1]
    rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1_score': f1_score(y_test, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, prob),
    })
    trained[name] = pipe

results = pd.DataFrame(rows).sort_values('f1_score', ascending=False)
results

## 4. Save Best Model

In [ ]:
Path('models').mkdir(exist_ok=True)
best_model_name = results.iloc[0]['model']
joblib.dump({'model_name': best_model_name, 'model': trained[best_model_name]}, 'models/best_spam_model.joblib')
print('Best model:', best_model_name)